In [1]:
# Install the necessary library (if MAVEN is in a standard format like JSON)
!pip install pandas

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import os
import pandas as pd
import json


# --- UPDATE THIS PATH ---
# Assuming the 'MAVEN Event Detection' folder is in your root MyDrive
MAVEN_FOLDER = '/content/drive/MyDrive/MAVEN Event Detection/'

# The MAVEN dataset typically uses 'train.jsonl', 'valid.jsonl', 'test.jsonl'
MAVEN_TRAIN_FILE = os.path.join(MAVEN_FOLDER, 'train.jsonl')
# You may need to adjust the sub-path if the files are nested, e.g., 'maven/train.jsonl'
# For now, let's assume 'train.jsonl' is directly in the 'MAVEN Event Detection' folder.
# ---

# 1. Load the .jsonl file line by line
maven_stories = []
try:
    with open(MAVEN_TRAIN_FILE, 'r') as f:
        for line in f:
            # Each line is a valid JSON object
            maven_stories.append(json.loads(line))

    # 2. Convert the list of JSON objects (stories/documents) into a DataFrame
    maven_df = pd.DataFrame(maven_stories)

    print("\n--- MAVEN Data Loaded Successfully (JSONL Format) ---")
    print(f"Total MAVEN Documents/Stories: {len(maven_df)}")
    print("Sample MAVEN Data Structure:")
    # Display the first few rows and column information
    print(maven_df.head())
    print("\nColumns and Data Types:")
    print(maven_df.info())

except FileNotFoundError:
    print(f"ERROR: train.jsonl not found at expected path: {MAVEN_TRAIN_FILE}")
    print("ACTION: Please ensure the MAVEN folder is named 'MAVEN Event Detection' and 'train.jsonl' is directly inside it, or update the MAVEN_TRAIN_FILE path.")
except Exception as e:
    print(f"An error occurred during MAVEN loading: {e}")


--- MAVEN Data Loaded Successfully (JSONL Format) ---
Total MAVEN Documents/Stories: 2913
Sample MAVEN Data Structure:
                                     title                                id  \
0  2006 Pangandaran earthquake and tsunami  8307a6b61b84d4eea42c1dd5e6e2cdba   
1             Battle of Santa Clara (1927)  387fe1dfe55067eb29e1fd4116d37af3   
2              Siege of Pondicherry (1793)  268c4763208c87ed7ebf55565c274d23   
3                        Battle of Leuthen  c95e68565081126b5c949117e423695a   
4           Glasgow St Enoch rail accident  3bec0b60c0940c5e46ee2cfc9504df92   

                                             content  \
0  [{'sentence': 'The 2006 Pangandaran earthquake...   
1  [{'sentence': 'The Battle of Santa Clara took ...   
2  [{'sentence': '"For other sieges with this nam...   
3  [{'sentence': 'The Battle of Leuthen was fough...   
4  [{'sentence': 'The Glasgow St Enoch rail accid...   

                                              events  \
0  [{'

In [4]:
import requests
import zipfile
import io
import pandas as pd
from datetime import date, timedelta
import os
import time
import re

# --- Configuration ---
# Switching target from GKG to GDELT Event File (which has reliable CAMEO codes).
GDELT_BASE_URL = "http://data.gdeltproject.org/events/"

# Define the date range (e.g., 3 months, ending on the date you provided)
START_DATE = date(2025, 7, 26)
END_DATE = date(2025, 10, 26)

# --- CRITICAL FIX: ADOPTING THE GDELT EVENT FILE SCHEMA (58 columns) ---
# Event files use the .export.CSV.zip format.
# FIX: The CAMEO Event Code is at Index 26, not 25.
COL_IDX_CAMEO_CODE = 26 # Index 26 contains the 3-digit CAMEO code for the event.
COL_IDX_URL = 57        # Index 57 contains the SOURCEURL.
MIN_REQUIRED_COLS = 58  # We now require 58 columns for the Event File format.

# --- Data Acquisition Loop ---
gdelt_data_list = []
current_date = START_DATE
day_counter = 0
skipped_days = 0

print("Starting GDELT Event File download and concatenation...")

while current_date <= END_DATE:
    date_str = current_date.strftime("%Y%m%d")
    # Using the GDELT Event file format: YYYYMMDD.export.CSV.zip
    file_name_zip = f"{date_str}.export.CSV.zip"
    file_name_csv = f"{date_str}.export.CSV"
    url = GDELT_BASE_URL + file_name_zip

    time.sleep(0.5)

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:
            # Read the zip file content from memory
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                # We assume the CSV filename inside the zip matches the pattern
                with z.open(file_name_csv) as csv_file:
                    # Read the daily CSV
                    daily_df = pd.read_csv(csv_file, sep='\t', header=None, encoding='utf-8',
                                            low_memory=False, on_bad_lines='skip')

                    # --- FINAL CHECK: Ensure the file matches the expected 58-column format ---
                    if daily_df.shape[1] != MIN_REQUIRED_COLS:
                        skipped_days += 1
                        print(f"| {date_str}: SKIPPED (Only {daily_df.shape[1]} cols, expected {MIN_REQUIRED_COLS} for Event File).")
                        current_date += timedelta(days=1)
                        continue
                    # ------------------------------------

                    gdelt_data_list.append(daily_df)
            day_counter += 1
            print(f"| {date_str}: Downloaded and processed. (Total: {day_counter} days)")
        else:
            # Skip if the file is not found (404) or other non-200 error
            pass

    except Exception as e:
        print(f"| {date_str}: ERROR during processing: {e}")

    current_date += timedelta(days=1)

print("\n--- Download Complete ---")
print(f"Total days skipped due to unexpected file format: {skipped_days}")

if not gdelt_data_list:
    raise Exception("No GDELT data was downloaded. Please verify the date range and URL connection.")

# Concatenate all daily DataFrames
gdelt_raw_df = pd.concat(gdelt_data_list, ignore_index=True)
print(f"Total raw GDELT records collected: {len(gdelt_raw_df)}")

# --- Filtering and Preparation ---

# Step 1: Initial count
print(f"| Step 1: Raw records after download: {len(gdelt_raw_df)}")

# Rename key columns for clarity using the 58-column indices
# NOTE: We are now targeting the correct index 26 for CAMEO Code
gdelt_raw_df.rename(columns={COL_IDX_URL: 'ArticleURL', COL_IDX_CAMEO_CODE: 'CAMEO_CODE'}, inplace=True)

# Filter 1: Drop rows where the ArticleURL is missing
gdelt_raw_df.dropna(subset=['ArticleURL'], inplace=True)
print(f"| Step 2: Records remaining after dropping null URLs: {len(gdelt_raw_df)}")

# Ensure 'CAMEO_CODE' column is a string before using str.contains
gdelt_raw_df['CAMEO_CODE'] = gdelt_raw_df['CAMEO_CODE'].fillna('').astype(str).str.strip()


# Filter 2: Select for your project's target themes using CAMEO codes.
# Target CAMEO families (2-digit prefixes) for Protest, Coercion, Conflict:
# 13 (Demonstrate), 14 (Protest), 17 (Coerce), 18 (Censure), 19 (Threaten), 20 (Fight)
target_prefixes = ['13', '14', '17', '18', '19', '20']
# The pattern is correct for 3- and 4-digit codes: e.g., ^14\d+$ matches 140, 1410, 1411, etc.
target_cameo_pattern = r'^(' + '|'.join(target_prefixes) + r')\d+$'

# Filter based on CAMEO code presence.
cameo_filter = gdelt_raw_df['CAMEO_CODE'].str.contains(target_cameo_pattern, case=False, regex=True)

gdelt_df_filtered = gdelt_raw_df[cameo_filter].copy()
print(f"| Step 3: Records remaining after CAMEO filtering: {len(gdelt_df_filtered)}")


# --- Sanity Check for Non-Zero Result ---
if len(gdelt_df_filtered) == 0 and len(gdelt_raw_df) > 0:
    print("\n*** CRITICAL ERROR: CAMEO FILTER FAILED AGAIN ***")
    print("Filter returned zero records, even after definitive index fix.")
    # Show the actual codes in the new target column (Index 26) for one final debug step
    print("Sample CAMEO Codes (Index 26):", gdelt_raw_df['CAMEO_CODE'].head(100).unique())
    print("************************************************\n")
# ----------------------------------------


# Filter 3: Select only the required columns and drop duplicate URLs
# We keep the CAMEO_CODE column for reference, but the key is 'ArticleURL'.
gdelt_df = gdelt_df_filtered[['ArticleURL', 'CAMEO_CODE']].drop_duplicates(subset=['ArticleURL']).reset_index(drop=True)
print(f"| Step 4: Unique articles remaining after deduplication: {len(gdelt_df)}")


# Final Step: Limit to the target sample size (100,000)
final_gdelt_df = gdelt_df.head(100000)

print("\n--- GDELT Data Filtered and Saved ---")
print(f"Final GDELT sample size for scraping: {len(final_gdelt_df)} articles.")

# Save the filtered metadata (with URLs) to your Google Drive
GDELT_SAVE_PATH = '/content/drive/MyDrive/NLP_Project_GDELT_Metadata_Filtered.csv'
final_gdelt_df.to_csv(GDELT_SAVE_PATH, index=False)
print(f"GDELT URL metadata saved to: {GDELT_SAVE_PATH}")

Starting GDELT Event File download and concatenation...
| 20250726: Downloaded and processed. (Total: 1 days)
| 20250727: Downloaded and processed. (Total: 2 days)
| 20250728: Downloaded and processed. (Total: 3 days)
| 20250729: Downloaded and processed. (Total: 4 days)
| 20250730: Downloaded and processed. (Total: 5 days)
| 20250731: Downloaded and processed. (Total: 6 days)
| 20250801: Downloaded and processed. (Total: 7 days)
| 20250802: Downloaded and processed. (Total: 8 days)
| 20250803: Downloaded and processed. (Total: 9 days)
| 20250804: Downloaded and processed. (Total: 10 days)
| 20250805: Downloaded and processed. (Total: 11 days)
| 20250806: Downloaded and processed. (Total: 12 days)
| 20250807: Downloaded and processed. (Total: 13 days)
| 20250808: Downloaded and processed. (Total: 14 days)
| 20250809: Downloaded and processed. (Total: 15 days)
| 20250810: Downloaded and processed. (Total: 16 days)
| 20250811: Downloaded and processed. (Total: 17 days)
| 20250812: Downlo

/tmp/ipython-input-1641715483.py:109: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cameo_filter = gdelt_raw_df['CAMEO_CODE'].str.contains(target_cameo_pattern, case=False, regex=True)


| Step 3: Records remaining after CAMEO filtering: 1624997
| Step 4: Unique articles remaining after deduplication: 607679

--- GDELT Data Filtered and Saved ---
Final GDELT sample size for scraping: 100000 articles.
GDELT URL metadata saved to: /content/drive/MyDrive/NLP_Project_GDELT_Metadata_Filtered.csv


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm # For tracking progress across 100,000 records

# --- Configuration ---
# Input file path created by the GDELT collector script
INPUT_METADATA_PATH = '/content/drive/MyDrive/NLP_Project_GDELT_Metadata_Filtered.csv'
# Output file path for the final scraped text
OUTPUT_DATA_PATH = '/content/drive/MyDrive/NLP_Project_Scraped_Articles.csv'
# Temporary save file for periodic saving
TEMP_OUTPUT_PATH = '/content/drive/MyDrive/NLP_Project_Scraped_Articles_TEMP.csv'

# Scraping settings
MAX_RETRIES = 3
TIMEOUT_SECONDS = 15
# Politeness delay to avoid overwhelming servers and getting blocked
SCRAPING_DELAY_SECONDS = 1
BATCH_SIZE = 1000 # Save progress every 1000 articles

# Standard user agent to mimic a common browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Setup robust session with retries for transient network failures
session = requests.Session()
retry_strategy = Retry(
    total=MAX_RETRIES,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504], # Retry on common server/rate limit errors
    allowed_methods=["HEAD", "GET", "OPTIONS"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("http://", adapter)
session.mount("https://", adapter)


def get_clean_article_text(html_content: str) -> str:
    """
    Parses HTML content using BeautifulSoup to extract and clean article text.
    It prioritizes common article-holding tags and content containers.
    """
    if not html_content:
        return ""

    soup = BeautifulSoup(html_content, 'html.parser')

    # Remove script, style, header, footer, and navigation elements
    for element in soup(["script", "style", "header", "footer", "nav", "aside", "form"]):
        element.decompose()

    # Define common selectors for article content
    content_containers = [
        # News-specific tags
        'article', 'main', 'body',
        # Common class names for main content
        '.story-body', '.article-body', '.article-content',
        '.post-content', '.entry-content', '#content'
    ]

    # Try to find the content using specific selectors
    for selector in content_containers:
        # Check if it's a tag or a class/id selector
        if selector.startswith('.') or selector.startswith('#'):
            element = soup.select_one(selector)
        else:
            element = soup.find(selector)

        if element and len(element.get_text(separator=' ', strip=True)) > 200:
            # If a large content block is found, extract text from it
            return element.get_text(separator='\n', strip=True)

    # Fallback: Just return the cleaned text of the entire body
    return soup.body.get_text(separator='\n', strip=True)


def scrape_articles_and_save():
    """Main function to load URLs, scrape articles, and save results."""
    print(f"Loading URLs from: {INPUT_METADATA_PATH}")
    try:
        df_metadata = pd.read_csv(INPUT_METADATA_PATH)
    except FileNotFoundError:
        print(f"ERROR: Input metadata file not found at {INPUT_METADATA_PATH}")
        return

    # Prepare list for scraped data and check if a temp file exists to resume
    scraped_data = []
    start_index = 0

    if os.path.exists(TEMP_OUTPUT_PATH):
        print(f"Resuming from temporary file: {TEMP_OUTPUT_PATH}")
        df_temp = pd.read_csv(TEMP_OUTPUT_PATH)
        scraped_data = df_temp.to_dict('records')

        # Calculate start index for resumption
        last_url_processed = df_temp['ArticleURL'].iloc[-1]
        try:
            start_index = df_metadata[df_metadata['ArticleURL'] == last_url_processed].index[0] + 1
            print(f"Starting scraping from index {start_index}...")
        except IndexError:
            # Should not happen, but handles case where last URL isn't in metadata
            print("Could not find last processed URL in metadata. Starting from index 0.")
            start_index = 0
            scraped_data = []


    print(f"Starting to scrape {len(df_metadata) - start_index} articles...")

    # Use tqdm for a clear progress bar
    for index, row in tqdm(df_metadata.iloc[start_index:].iterrows(), total=len(df_metadata.iloc[start_index:]), initial=start_index, desc="Scraping"):
        url = row['ArticleURL']
        cameo_code = row['CAMEO_CODE']
        article_text = ""
        status_code = -1

        try:
            # Fetch the page content
            response = session.get(url, headers=HEADERS, timeout=TIMEOUT_SECONDS, allow_redirects=True)
            status_code = response.status_code

            # Only process if request was successful (200-299)
            if 200 <= status_code < 300:
                article_text = get_clean_article_text(response.text)

                # Simple check for content length (e.g., must be longer than a sentence)
                if len(article_text) < 150:
                    article_text = f"[CONTENT TOO SHORT OR FAILED EXTRACTION] - Scraped length: {len(article_text)}"

            elif status_code in [404, 410]:
                article_text = f"[DEAD LINK] - Status: {status_code}"
            else:
                article_text = f"[REQUEST FAILED] - Status: {status_code}"

        except requests.exceptions.RequestException as e:
            # Handle all request errors (e.g., timeout, connection error)
            article_text = f"[NETWORK ERROR] - {type(e).__name__}"

        except Exception as e:
            # Catch any other unexpected errors
            article_text = f"[UNEXPECTED ERROR] - {type(e).__name__}"

        # Append result to the list
        scraped_data.append({
            'ArticleURL': url,
            'CAMEO_CODE': cameo_code,
            'StatusCode': status_code,
            'ArticleText': article_text
        })

        # Periodic saving (Batch saving)
        if (index + 1) % BATCH_SIZE == 0:
            df_temp_save = pd.DataFrame(scraped_data)
            df_temp_save.to_csv(TEMP_OUTPUT_PATH, index=False)
            print(f"\n--- Progress Saved ({len(scraped_data)} records) ---")

        # Politeness delay
        time.sleep(SCRAPING_DELAY_SECONDS)

    # --- Final Save ---
    df_final = pd.DataFrame(scraped_data)
    df_final.to_csv(OUTPUT_DATA_PATH, index=False)

    # Remove temp file after successful completion
    if os.path.exists(TEMP_OUTPUT_PATH):
        os.remove(TEMP_OUTPUT_PATH)


    # Final Summary
    successful_count = df_final[~df_final['ArticleText'].str.startswith('[')].shape[0]
    failed_count = df_final[df_final['ArticleText'].str.startswith('[')].shape[0]

    print("\n--- Scraping Complete ---")
    print(f"Total processed records: {len(df_final)}")
    print(f"Successfully scraped articles: {successful_count}")
    print(f"Failed/Skipped records: {failed_count}")
    print(f"Final data saved to: {OUTPUT_DATA_PATH}")


if __name__ == '__main__':
    # Increase the recursion limit for deep HTML parsing
    import sys
    sys.setrecursionlimit(2000)

    scrape_articles_and_save()

Loading URLs from: /content/drive/MyDrive/NLP_Project_GDELT_Metadata_Filtered.csv
Starting to scrape 100000 articles...


Scraping:   1%|          | 999/100000 [38:11<39:51:12,  1.45s/it]


--- Progress Saved (1000 records) ---


Scraping:   2%|▏         | 1999/100000 [1:12:47<57:59:33,  2.13s/it]


--- Progress Saved (2000 records) ---


...

Scraping:  34%|███▍      | 33999/100000 [22:51:59<30:12:55,  1.65s/it]


--- Progress Saved (34000 records) ---


Scraping:  35%|███▍      | 34999/100000 [23:35:56<39:52:51,  2.21s/it]


--- Progress Saved (35000 records) ---


Scraping:  35%|███▌      | 35374/100000 [23:50:07<34:01:31,  1.90s/it]